In [1]:
%matplotlib inline

# eighth_attempt.ipynb -- Diabetic Retinopathy Detection

**Changes vs seventh_attempt.ipynb (K-series):**
- **(K-1) Revert TTA to 4-pass**: Drop rot90 variants — original + hflip + vflip + rot180 only. 8-pass hurt both models (custom 0.756 vs 0.770, ft 0.768 vs 0.781). rot90cw/rot90ccw introduce distribution shift not seen during training.
- **(K-2) Multi-architecture custom ensemble**: Replace 2-seed SE-CustomNetV2 with SE-CustomNetV2 (channel attention) + CustomNetV3 (residual skip connections). Different inductive biases → complementary errors → stronger ensemble.
- **(K-3) Val-based ensemble weight**: Grid-search w ∈ {0.1..0.9} on val; output_custom.csv = w × SE-CustomNetV2 + (1-w) × CustomNetV3. Both are scratch models — any blend is valid for CUSTOM.
- **(K-4) ft model: sixth_attempt Stage 1+2 only**: Remove Stage 3 (denseblock2 unfreezing hurt: 0.768 vs 0.781). Copy sixth_attempt DenseNet121 exactly.
- **(K-5) Fix cudnn setup**: Replace `cudnn.enabled = False` (disabled cuDNN entirely) with `cudnn.deterministic = True; cudnn.benchmark = False`.

**CUSTOM validity**: output_custom.csv = blend of SE-CustomNetV2 and CustomNetV3 — both trained from scratch, no pretrained weights.

## 1. Imports & Setup

In [2]:
from __future__ import print_function, division
import os, csv
import torch
import pandas as pd
from skimage import io, transform, util, color
from sklearn import metrics
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, utils, models
from torchvision.models import densenet121, DenseNet121_Weights
import torchvision.transforms.functional as TF
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim import lr_scheduler
import time
import copy
from PIL import Image
from zipfile import ZipFile
import random
import numpy.random as npr
import cv2
import warnings

warnings.filterwarnings('ignore')
random.seed(42)
npr.seed(42)
torch.manual_seed(42)
torch.backends.cudnn.deterministic = True   # K-5: was cudnn.enabled=False in seventh_attempt
torch.backends.cudnn.benchmark = False

plt.ion()
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

cuda:0


In [3]:
DATA_ROOT = '/kaggle/input/datasets/mariamuozperez/lab5-cv/LS5_CV_2025_2026_DB_Retinopathy'

In [4]:
# Run once to extract data, then comment out
# import zipfile
# with zipfile.ZipFile('./db.zip', 'r') as z:
#     z.extractall('./data')

## 2. Dataset

In [5]:
class RetinopathyDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None, maxSize=0):
        self.dataset = pd.read_csv(csv_file, header=0,
                                   dtype={'id': str, 'eye': int, 'label': int})
        if maxSize > 0:
            idx = np.random.RandomState(seed=42).permutation(range(len(self.dataset)))
            self.dataset = self.dataset.iloc[idx[:maxSize]].reset_index(drop=True)
        self.root_dir = root_dir
        self.img_dir  = os.path.join(root_dir, 'images')
        self.transform = transform
        self.levels  = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative DR']
        self.classes = ['No DR', 'DR']

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()
        img_name = os.path.join(self.img_dir, self.dataset.id[idx] + '.jpg')
        image = io.imread(img_name)
        if self.dataset.eye[idx] == 1:
            image = image[:, ::-1, :]
        sample = {
            'image': image,
            'eye':   self.dataset.eye[idx],
            'label': (self.dataset.label[idx] > 0).astype(dtype=np.int64)
        }
        if self.transform:
            sample = self.transform(sample)
        return sample

## 3. Transforms

In [6]:
class CropByEye(object):
    def __init__(self, threshold, border):
        self.threshold = threshold
        self.border = (border, border) if isinstance(border, int) else border

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        imgray = color.rgb2gray(image)
        _, mask = cv2.threshold(imgray, self.threshold, 1, cv2.THRESH_BINARY)
        sidx = np.nonzero(mask)
        if len(sidx[0]) < 20:
            return {'image': image, 'eye': eye, 'label': label}
        minx = np.maximum(sidx[1].min() - self.border[1], 0)
        maxx = np.minimum(sidx[1].max() + 1 + self.border[1], w)
        miny = np.maximum(sidx[0].min() - self.border[0], 0)
        maxy = np.minimum(sidx[0].max() + 1 + self.border[1], h)
        image = image[miny:maxy, minx:maxx, ...]
        return {'image': image, 'eye': eye, 'label': label}


class BenGraham(object):
    def __init__(self, sigmaX=10):
        self.sigmaX = sigmaX

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        if image.dtype == np.uint8:
            img_u8 = image
        else:
            img_u8 = (np.clip(image, 0, 1) * 255).astype(np.uint8)
        blurred  = cv2.GaussianBlur(img_u8, (0, 0), self.sigmaX)
        enhanced = cv2.addWeighted(img_u8, 4, blurred, -4, 128)
        enhanced = np.clip(enhanced, 0, 255).astype(np.uint8)
        mask = np.zeros(enhanced.shape, dtype=np.uint8)
        h, w = enhanced.shape[:2]
        cv2.circle(mask, (w // 2, h // 2), int(0.9 * min(h, w) / 2), (1, 1, 1), -1, 8, 0)
        enhanced = enhanced * mask + 128 * (1 - mask)
        return {'image': enhanced.astype(np.float32) / 255.0, 'eye': eye, 'label': label}


class Rescale(object):
    def __init__(self, output_size):
        self.output_size = output_size

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        if isinstance(self.output_size, int):
            new_h = self.output_size * h / w if h > w else self.output_size
            new_w = self.output_size if h > w else self.output_size * w / h
        else:
            new_h, new_w = self.output_size
        image = transform.resize(image, (int(new_h), int(new_w)))
        return {'image': image, 'eye': eye, 'label': label}


class RandomCrop(object):
    def __init__(self, output_size):
        self.output_size = (output_size, output_size) if isinstance(output_size, int) else output_size

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        new_h, new_w = self.output_size
        top  = np.random.randint(0, h - new_h) if h > new_h else 0
        left = np.random.randint(0, w - new_w) if w > new_w else 0
        image = image[top:top + new_h, left:left + new_w]
        return {'image': image, 'eye': eye, 'label': label}


class CenterCrop(object):
    def __init__(self, output_size):
        self.output_size = (output_size, output_size) if isinstance(output_size, int) else output_size

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        new_h, new_w = self.output_size
        top  = int((h - new_h) / 2) if h > new_h else 0
        left = int((w - new_w) / 2) if w > new_w else 0
        image = image[top:top + new_h, left:left + new_w]
        return {'image': image, 'eye': eye, 'label': label}


class ToTensor(object):
    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        image = torch.from_numpy(image.transpose((2, 0, 1)))
        label = torch.tensor(label, dtype=torch.long)
        return {'image': image, 'eye': eye, 'label': label}


class Normalize(object):
    def __init__(self, mean, std):
        self.mean = np.array(mean)
        self.std  = np.array(std)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        dtype = image.dtype
        mean = torch.as_tensor(self.mean, dtype=dtype, device=image.device)
        std  = torch.as_tensor(self.std,  dtype=dtype, device=image.device)
        image.sub_(mean[:, None, None]).div_(std[:, None, None])
        return {'image': image, 'eye': eye, 'label': label}


class TVCenterCrop(object):
    def __init__(self, size):
        self.CC = transforms.CenterCrop(size)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        pil = Image.fromarray(util.img_as_ubyte(image))
        image = util.img_as_float(np.asarray(self.CC(pil)))
        return {'image': image, 'eye': eye, 'label': label}


class TVRandomHorizontalFlip(object):
    def __init__(self, p=0.5):
        self.flip = transforms.RandomHorizontalFlip(p=p)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        pil = Image.fromarray(util.img_as_ubyte(image))
        image = util.img_as_float(np.asarray(self.flip(pil)))
        return {'image': image, 'eye': eye, 'label': label}


class TVRandomRotation(object):
    def __init__(self, degrees=15):
        self.rotate = transforms.RandomRotation(degrees=degrees)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        pil = Image.fromarray(util.img_as_ubyte(image))
        image = util.img_as_float(np.asarray(self.rotate(pil)))
        return {'image': image, 'eye': eye, 'label': label}


class TVColorJitter(object):
    def __init__(self, brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05):
        self.jitter = transforms.ColorJitter(
            brightness=brightness, contrast=contrast,
            saturation=saturation, hue=hue)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        pil = Image.fromarray(util.img_as_ubyte(image))
        image = util.img_as_float(np.asarray(self.jitter(pil)))
        return {'image': image, 'eye': eye, 'label': label}

## 4. Data Pipelines & DataLoaders

In [7]:
pixel_mean = [0.485, 0.456, 0.406]
pixel_std  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    CropByEye(0.10, 1),
    BenGraham(sigmaX=10),
    Rescale(256),
    TVRandomHorizontalFlip(p=0.5),
    TVRandomRotation(degrees=15),
    TVColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    RandomCrop(224),
    ToTensor(),
    Normalize(mean=pixel_mean, std=pixel_std),
])

eval_transform = transforms.Compose([
    CropByEye(0.10, 1),
    BenGraham(sigmaX=10),
    Rescale(256),
    CenterCrop(224),
    ToTensor(),
    Normalize(mean=pixel_mean, std=pixel_std),
])

train_dataset = RetinopathyDataset(
    csv_file=os.path.join(DATA_ROOT, 'train.csv'),
    root_dir=DATA_ROOT, maxSize=0, transform=train_transform)
val_dataset = RetinopathyDataset(
    csv_file=os.path.join(DATA_ROOT, 'val.csv'),
    root_dir=DATA_ROOT, transform=eval_transform)
test_dataset = RetinopathyDataset(
    csv_file=os.path.join(DATA_ROOT, 'test.csv'),
    root_dir=DATA_ROOT, transform=eval_transform)
print(f'Train: {len(train_dataset)}  Val: {len(val_dataset)}  Test: {len(test_dataset)}')

Train: 2000  Val: 500  Test: 1000


In [8]:
_sample = train_dataset[0]
_img = _sample['image'].numpy().transpose(1, 2, 0)
print(f'Pipeline output -- dtype: {_img.dtype}, min: {_img.min():.3f}, max: {_img.max():.3f}, mean: {_img.mean():.3f}')
assert abs(_img.mean()) < 0.3, f'BenGraham all-gray bug! mean={_img.mean():.3f}'
print('Sanity check passed.')

Pipeline output -- dtype: float64, min: -1.827, max: 2.152, mean: 0.283
Sanity check passed.


In [9]:
train_labels_bin_for_sampler = (train_dataset.dataset['label'].values > 0).astype(int)
class_counts   = np.bincount(train_labels_bin_for_sampler)
sample_weights = np.where(train_labels_bin_for_sampler == 1,
                          1.0 / class_counts[1], 1.0 / class_counts[0])
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.float),
    num_samples=len(train_dataset), replacement=True)

train_dataloader = DataLoader(train_dataset, batch_size=64,  sampler=sampler,  num_workers=0)
val_dataloader   = DataLoader(val_dataset,   batch_size=256, shuffle=False, num_workers=0)
test_dataloader  = DataLoader(test_dataset,  batch_size=256, shuffle=False, num_workers=0)

train_labels_bin = (train_dataset.dataset['label'].values > 0).astype(int)
n_neg = int((train_labels_bin == 0).sum())
n_pos = int((train_labels_bin == 1).sum())
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float).to(device)
print(f'No-DR: {n_neg}  DR: {n_pos}  pos_weight: {pos_weight.item():.3f}')

criterion      = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
image_datasets = {'train': train_dataset, 'val': val_dataset}
dataloaders    = {'train': train_dataloader, 'val': val_dataloader}
dataset_sizes  = {'train': len(train_dataset), 'val': len(val_dataset)}
class_names    = train_dataset.classes

No-DR: 1468  DR: 532  pos_weight: 2.759


## 5. Training & Evaluation Utilities

In [10]:
def train_model(model, criterion, optimizer, scheduler, num_epochs=25, patience=7, label_smoothing=0.0):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_auc, best_epoch, no_improve = 0.0, -1, 0

    for epoch in range(num_epochs):
        print('Epoch {}/{}'.format(epoch, num_epochs - 1))
        print('-' * 10)
        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()
            numSamples = dataset_sizes[phase]
            outputs_m  = np.zeros((numSamples,), dtype=float)
            labels_m   = np.zeros((numSamples,), dtype=int)
            running_loss, contSamples = 0.0, 0
            for sample in dataloaders[phase]:
                inputs    = sample['image'].to(device).float()
                labels    = sample['label'].to(device).float()
                batchSize = labels.shape[0]
                optimizer.zero_grad()
                with torch.set_grad_enabled(phase == 'train'):
                    logits = model(inputs).flatten()
                    if label_smoothing > 0.0 and phase == 'train':
                        labels_ls = labels * (1 - label_smoothing) + label_smoothing / 2.0
                        loss = criterion(logits, labels_ls)
                    else:
                        loss = criterion(logits, labels)
                    scores = torch.sigmoid(logits).detach()
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                running_loss += loss.item() * batchSize
                outputs_m[contSamples:contSamples + batchSize] = scores.cpu().numpy()
                labels_m [contSamples:contSamples + batchSize] = labels.cpu().numpy()
                contSamples += batchSize
            if phase == 'train':
                scheduler.step()
            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_auc  = metrics.roc_auc_score(labels_m, outputs_m)
            print('{} Loss: {:.4f}  AUC: {:.4f}'.format(phase, epoch_loss, epoch_auc))
            if phase == 'val':
                if epoch_auc > best_auc:
                    best_auc, best_epoch, no_improve = epoch_auc, epoch, 0
                    best_model_wts = copy.deepcopy(model.state_dict())
                else:
                    no_improve += 1
                    if no_improve >= patience:
                        print(f'Early stopping: no improvement for {patience} epochs.')
                        model.load_state_dict(best_model_wts)
                        return model
        print()
    elapsed = time.time() - since
    print('Training complete in {:.0f}m {:.0f}s'.format(elapsed // 60, elapsed % 60))
    print('Best model: epoch {:d}  val AUC: {:.4f}'.format(best_epoch, best_auc))
    model.load_state_dict(best_model_wts)
    return model

In [11]:
# K-1: TTA reverted to 4-pass (original + hflip + vflip + rot180).
# 8-pass (seventh_attempt) introduced distribution shift via rot90 variants.

def eval_val_auc(model, name, tta=False):
    model.eval()
    n = len(val_dataset)
    scores_m = np.zeros((n, 1), dtype=float)
    labels_m = np.zeros((n,), dtype=int)
    cont = 0
    with torch.no_grad():
        for sample in val_dataloader:
            inputs = sample['image'].to(device).float()
            bs = inputs.shape[0]
            if tta:
                s1 = torch.sigmoid(model(inputs))
                s2 = torch.sigmoid(model(torch.flip(inputs, dims=[3])))    # hflip
                s3 = torch.sigmoid(model(torch.flip(inputs, dims=[2])))    # vflip
                s4 = torch.sigmoid(model(torch.flip(inputs, dims=[2, 3]))) # rot180
                out = (s1 + s2 + s3 + s4) / 4.0
            else:
                out = torch.sigmoid(model(inputs))
            scores_m[cont:cont + bs, :] = out.cpu().numpy()
            labels_m[cont:cont + bs]     = sample['label'].numpy()
            cont += bs
    auc    = metrics.roc_auc_score(labels_m, scores_m)
    suffix = ' (TTA-4)' if tta else ''
    print(f'{name}{suffix}  --  val AUC: {auc:.4f}')
    return auc


def get_val_scores(model, tta=False):
    """Return (scores_array, labels_array) for val set — used for ensemble weight search."""
    model.eval()
    n = len(val_dataset)
    scores_m = np.zeros((n, 1), dtype=float)
    labels_m = np.zeros((n,), dtype=int)
    cont = 0
    with torch.no_grad():
        for sample in val_dataloader:
            inputs = sample['image'].to(device).float()
            bs = inputs.shape[0]
            if tta:
                s1 = torch.sigmoid(model(inputs))
                s2 = torch.sigmoid(model(torch.flip(inputs, dims=[3])))
                s3 = torch.sigmoid(model(torch.flip(inputs, dims=[2])))
                s4 = torch.sigmoid(model(torch.flip(inputs, dims=[2, 3])))
                out = (s1 + s2 + s3 + s4) / 4.0
            else:
                out = torch.sigmoid(model(inputs))
            scores_m[cont:cont + bs, :] = out.cpu().numpy()
            labels_m[cont:cont + bs]     = sample['label'].numpy()
            cont += bs
    return scores_m, labels_m


def test_model(model, tta=False):
    model.eval()
    n = len(test_dataset)
    outputs_m = np.zeros((n, 1), dtype=float)
    cont = 0
    with torch.no_grad():
        for sample in test_dataloader:
            inputs = sample['image'].to(device).float()
            bs = inputs.shape[0]
            if tta:
                s1 = torch.sigmoid(model(inputs))
                s2 = torch.sigmoid(model(torch.flip(inputs, dims=[3])))
                s3 = torch.sigmoid(model(torch.flip(inputs, dims=[2])))
                s4 = torch.sigmoid(model(torch.flip(inputs, dims=[2, 3])))
                out = (s1 + s2 + s3 + s4) / 4.0
            else:
                out = torch.sigmoid(model(inputs))
            outputs_m[cont:cont + bs, :] = out.cpu().numpy()
            cont += bs
    return outputs_m

---
## 6. SE-CustomNetV2 (CUSTOM — Architecture A)

**(K-2)** Channel-attention model. SEBlock after Conv-BN-ReLU in every _block. r=8.

In [12]:
class SEBlock(nn.Module):
    def __init__(self, channels, r=8):
        super().__init__()
        mid = max(channels // r, 4)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc  = nn.Sequential(
            nn.Linear(channels, mid),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        w = self.gap(x).flatten(1)
        w = self.fc(w).view(x.size(0), x.size(1), 1, 1)
        return x * w

In [13]:
class CustomNetV2(nn.Module):
    """SE-CustomNetV2: 5-block sequential CNN with channel-attention (SEBlock) per block."""

    def __init__(self):
        super().__init__()

        def _block(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(cout),
                nn.ReLU(inplace=True),
                SEBlock(cout),
                nn.MaxPool2d(2, 2),
            )

        self.features = nn.Sequential(
            _block(3,   32),
            _block(32,  64),
            _block(64,  128),
            _block(128, 256),
            _block(256, 256),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.2),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.4),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x).flatten(1)
        return self.classifier(x)

In [14]:
_net = CustomNetV2().to(device)
_inp = next(iter(train_dataloader))['image'].to(device).float()
with torch.no_grad():
    _out = _net(_inp)
print(f'SE-CustomNetV2  Input: {_inp.shape}  Output: {_out.shape}')
total  = sum(p.numel() for p in _net.parameters())
se_tot = sum(p.numel() for m in _net.modules() if isinstance(m, SEBlock) for p in m.parameters())
print(f'SE-CustomNetV2 -- total params: {total:,}  SE params: {se_tot:,}')
del _net, _inp, _out

SE-CustomNetV2  Input: torch.Size([64, 3, 224, 224])  Output: torch.Size([64, 1])
SE-CustomNetV2 -- total params: 1,051,229  SE params: 38,972


---
## 7. CustomNetV3 (CUSTOM — Architecture B)

**(K-2)** Residual-connection model. Each block: Conv2d(3×3)→BN + 1×1-projection-skip → ReLU → MaxPool.
Diversity: SE-V2 does channel recalibration; V3 does residual learning (gradient flows directly to early layers,
preserving low-level vessel/lesion details).
Same channel progression (3→32→64→128→256→256), GAP, and classifier head as SE-CustomNetV2.

In [15]:
class ResBlock(nn.Module):
    """Downsampling block with residual shortcut. 1x1 projection when cin != cout."""

    def __init__(self, cin, cout):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(cin, cout, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(cout),
        )
        self.skip = nn.Sequential(
            nn.Conv2d(cin, cout, kernel_size=1, bias=False),
            nn.BatchNorm2d(cout),
        ) if cin != cout else nn.Identity()
        self.relu = nn.ReLU(inplace=True)
        self.pool = nn.MaxPool2d(2, 2)

    def forward(self, x):
        out = self.main(x) + self.skip(x)
        return self.pool(self.relu(out))


class CustomNetV3(nn.Module):
    """CustomNetV3: 5-block residual CNN. Comparable param count to SE-CustomNetV2."""

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            ResBlock(3,   32),
            ResBlock(32,  64),
            ResBlock(64,  128),
            ResBlock(128, 256),
            ResBlock(256, 256),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.2),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.4),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x).flatten(1)
        return self.classifier(x)

In [16]:
_net3 = CustomNetV3().to(device)
_inp3 = next(iter(train_dataloader))['image'].to(device).float()
with torch.no_grad():
    _out3 = _net3(_inp3)
print(f'CustomNetV3  Input: {_inp3.shape}  Output: {_out3.shape}')
total3 = sum(p.numel() for p in _net3.parameters())
print(f'CustomNetV3 -- total params: {total3:,}')
del _net3, _inp3, _out3

CustomNetV3  Input: torch.Size([64, 3, 224, 224])  Output: torch.Size([64, 1])
CustomNetV3 -- total params: 1,056,321


---
## 8. Train SE-CustomNetV2 (Architecture A, seed=42)

In [17]:
random.seed(42); npr.seed(42)
torch.manual_seed(42); torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

model_se    = CustomNetV2().to(device)
optimizer_se = optim.AdamW(model_se.parameters(), lr=1e-3, weight_decay=5e-3)
scheduler_se = lr_scheduler.CosineAnnealingLR(optimizer_se, T_max=50)
model_se = train_model(model_se, criterion, optimizer_se, scheduler_se,
                       num_epochs=50, patience=10, label_smoothing=0.0)

torch.save(model_se.state_dict(), 'best_se_customnetv2.pth')
print('Saved: best_se_customnetv2.pth')
auc_se = eval_val_auc(model_se, 'SE-CustomNetV2', tta=True)

Epoch 0/49
----------
train Loss: 1.1131  AUC: 0.5062
val Loss: 1.1876  AUC: 0.4912

Epoch 1/49
----------
train Loss: 1.0899  AUC: 0.5142
val Loss: 1.1452  AUC: 0.5458

Epoch 2/49
----------
train Loss: 1.0902  AUC: 0.5329
val Loss: 1.2460  AUC: 0.6116

Epoch 3/49
----------
train Loss: 1.0836  AUC: 0.5436
val Loss: 1.2195  AUC: 0.6422

Epoch 4/49
----------
train Loss: 1.0552  AUC: 0.5987
val Loss: 1.1816  AUC: 0.6819

Epoch 5/49
----------
train Loss: 1.0519  AUC: 0.6363
val Loss: 1.0288  AUC: 0.6950

Epoch 6/49
----------
train Loss: 1.0343  AUC: 0.6471
val Loss: 1.3343  AUC: 0.6940

Epoch 7/49
----------
train Loss: 0.9972  AUC: 0.6895
val Loss: 0.9936  AUC: 0.7064

Epoch 8/49
----------
train Loss: 0.9825  AUC: 0.7107
val Loss: 0.9854  AUC: 0.6948

Epoch 9/49
----------
train Loss: 1.0224  AUC: 0.6771
val Loss: 0.9740  AUC: 0.7255

Epoch 10/49
----------
train Loss: 1.0011  AUC: 0.6901
val Loss: 1.0258  AUC: 0.7154

Epoch 11/49
----------
train Loss: 0.9719  AUC: 0.7138
val Loss:

---
## 9. Train CustomNetV3 (Architecture B, seed=42)

In [18]:
random.seed(42); npr.seed(42)
torch.manual_seed(42); torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

model_v3     = CustomNetV3().to(device)
optimizer_v3 = optim.AdamW(model_v3.parameters(), lr=1e-3, weight_decay=5e-3)
scheduler_v3 = lr_scheduler.CosineAnnealingLR(optimizer_v3, T_max=50)
model_v3 = train_model(model_v3, criterion, optimizer_v3, scheduler_v3,
                       num_epochs=50, patience=10, label_smoothing=0.0)

torch.save(model_v3.state_dict(), 'best_customnetv3.pth')
print('Saved: best_customnetv3.pth')
auc_v3 = eval_val_auc(model_v3, 'CustomNetV3', tta=True)

Epoch 0/49
----------
train Loss: 1.1414  AUC: 0.5152
val Loss: 1.2311  AUC: 0.6165

Epoch 1/49
----------
train Loss: 1.1093  AUC: 0.5116
val Loss: 1.0751  AUC: 0.5715

Epoch 2/49
----------
train Loss: 1.0862  AUC: 0.5600
val Loss: 1.2776  AUC: 0.6679

Epoch 3/49
----------
train Loss: 1.0583  AUC: 0.5951
val Loss: 1.0546  AUC: 0.6623

Epoch 4/49
----------
train Loss: 1.0335  AUC: 0.6429
val Loss: 1.0328  AUC: 0.6784

Epoch 5/49
----------
train Loss: 1.0184  AUC: 0.6611
val Loss: 1.1866  AUC: 0.6833

Epoch 6/49
----------
train Loss: 1.0392  AUC: 0.6477
val Loss: 1.1444  AUC: 0.6792

Epoch 7/49
----------
train Loss: 1.0006  AUC: 0.6771
val Loss: 1.3389  AUC: 0.7059

Epoch 8/49
----------
train Loss: 1.0180  AUC: 0.6638
val Loss: 1.3275  AUC: 0.6999

Epoch 9/49
----------
train Loss: 1.0016  AUC: 0.6838
val Loss: 1.0245  AUC: 0.7049

Epoch 10/49
----------
train Loss: 0.9792  AUC: 0.7110
val Loss: 1.1791  AUC: 0.7031

Epoch 11/49
----------
train Loss: 1.0094  AUC: 0.6803
val Loss:

---
## 10. Custom Ensemble Weight Search (K-3)

Grid-search w ∈ {0.1..0.9} on val. Both models are scratch-only → any w is valid for CUSTOM category.

output_custom.csv = w × SE-CustomNetV2 + (1-w) × CustomNetV3

In [19]:
val_scores_se, val_labels = get_val_scores(model_se, tta=True)
val_scores_v3, _          = get_val_scores(model_v3, tta=True)

print(f'SE-CustomNetV2 val AUC (TTA-4): {metrics.roc_auc_score(val_labels, val_scores_se):.4f}')
print(f'CustomNetV3    val AUC (TTA-4): {metrics.roc_auc_score(val_labels, val_scores_v3):.4f}')

best_w, best_ensemble_auc = 0.5, 0.0
print('\nEnsemble weight search:')
for w in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    blended = w * val_scores_se + (1 - w) * val_scores_v3
    auc = metrics.roc_auc_score(val_labels, blended)
    print(f'  w={w:.1f}  ensemble val AUC: {auc:.4f}')
    if auc > best_ensemble_auc:
        best_ensemble_auc, best_w = auc, w

print(f'\nBest w={best_w:.1f}  ensemble val AUC: {best_ensemble_auc:.4f}')
auc_custom_final = best_ensemble_auc

SE-CustomNetV2 val AUC (TTA-4): 0.7716
CustomNetV3    val AUC (TTA-4): 0.7656

Ensemble weight search:
  w=0.1  ensemble val AUC: 0.7704
  w=0.2  ensemble val AUC: 0.7730
  w=0.3  ensemble val AUC: 0.7744
  w=0.4  ensemble val AUC: 0.7757
  w=0.5  ensemble val AUC: 0.7758
  w=0.6  ensemble val AUC: 0.7757
  w=0.7  ensemble val AUC: 0.7753
  w=0.8  ensemble val AUC: 0.7738
  w=0.9  ensemble val AUC: 0.7729

Best w=0.5  ensemble val AUC: 0.7758


---
## 11. DenseNet121 — Stage 1 (FINE-TUNING category)

**(K-4)** Exact copy of sixth_attempt Stage 1+2. No Stage 3 (denseblock2 unfreezing hurt in seventh_attempt).

Unfreeze denseblock4 + norm5. Head: Dropout(0.5) -> Linear(1024,1).
AdamW lr=3e-4 wd=1e-2, CosineAnnealingLR(T_max=30), 30 epochs, patience=7, label_smoothing=0.05.

In [20]:
ftNet = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)
for param in ftNet.parameters():
    param.requires_grad = False
ftNet.classifier = nn.Sequential(nn.Dropout(p=0.5), nn.Linear(1024, 1))
for param in ftNet.features.denseblock4.parameters():
    param.requires_grad = True
for param in ftNet.features.norm5.parameters():
    param.requires_grad = True
ftNet = ftNet.to(device)
trainable = sum(p.numel() for p in ftNet.parameters() if p.requires_grad)
total     = sum(p.numel() for p in ftNet.parameters())
print(f'DenseNet121  Trainable: {trainable:,} / {total:,} params')

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 198MB/s]


DenseNet121  Trainable: 2,161,153 / 6,954,881 params


In [21]:
optimizer_ft_s1 = optim.AdamW(
    filter(lambda p: p.requires_grad, ftNet.parameters()),
    lr=3e-4, weight_decay=1e-2)
scheduler_ft_s1 = lr_scheduler.CosineAnnealingLR(optimizer_ft_s1, T_max=30)

In [22]:
random.seed(42); npr.seed(42); torch.manual_seed(42); torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
ftNet = train_model(ftNet, criterion, optimizer_ft_s1, scheduler_ft_s1,
                    num_epochs=30, patience=7, label_smoothing=0.05)

Epoch 0/29
----------
train Loss: 1.0754  AUC: 0.6384
val Loss: 0.9852  AUC: 0.7026

Epoch 1/29
----------
train Loss: 0.9789  AUC: 0.7358
val Loss: 0.9965  AUC: 0.7497

Epoch 2/29
----------
train Loss: 0.8768  AUC: 0.8053
val Loss: 0.9887  AUC: 0.7407

Epoch 3/29
----------
train Loss: 0.8361  AUC: 0.8342
val Loss: 0.9167  AUC: 0.7360

Epoch 4/29
----------
train Loss: 0.7961  AUC: 0.8556
val Loss: 0.8776  AUC: 0.7413

Epoch 5/29
----------
train Loss: 0.7492  AUC: 0.8794
val Loss: 0.9531  AUC: 0.7194

Epoch 6/29
----------
train Loss: 0.7375  AUC: 0.8830
val Loss: 1.0937  AUC: 0.7211

Epoch 7/29
----------
train Loss: 0.6921  AUC: 0.9030
val Loss: 1.0091  AUC: 0.7468

Epoch 8/29
----------
train Loss: 0.6490  AUC: 0.9218
val Loss: 0.9910  AUC: 0.7222
Early stopping: no improvement for 7 epochs.


In [23]:
torch.save(ftNet.state_dict(), 'best_densenet121_s1.pth')
print('Saved: best_densenet121_s1.pth')
auc_ft_s1     = eval_val_auc(ftNet, 'DenseNet121 Stage 1', tta=False)
auc_ft_s1_tta = eval_val_auc(ftNet, 'DenseNet121 Stage 1', tta=True)

Saved: best_densenet121_s1.pth
DenseNet121 Stage 1  --  val AUC: 0.7497
DenseNet121 Stage 1 (TTA-4)  --  val AUC: 0.7731


---
## 12. DenseNet121 — Stage 2: Unfreeze denseblock3 + transition3

3-epoch linear warmup + CosineAnnealingLR(T_max=12). 15 epochs, patience=5.
Reverts to Stage 1 if Stage 2 val AUC (TTA) is lower.
Gate: only proceeds if Stage 1 + TTA val AUC >= 0.752.

In [24]:
if auc_ft_s1_tta >= 0.752:
    print(f'Stage 1 TTA AUC {auc_ft_s1_tta:.4f} >= 0.752. Proceeding to Stage 2.')

    ftNet.load_state_dict(torch.load('best_densenet121_s1.pth', map_location=device))
    for param in ftNet.features.denseblock3.parameters():
        param.requires_grad = True
    for param in ftNet.features.transition3.parameters():
        param.requires_grad = True
    trainable_s2 = sum(p.numel() for p in ftNet.parameters() if p.requires_grad)
    print(f'Stage 2 trainable params: {trainable_s2:,}')

    optimizer_ft_s2 = optim.AdamW([
        {'params': ftNet.features.denseblock3.parameters(),  'lr': 3e-5},
        {'params': ftNet.features.transition3.parameters(),  'lr': 3e-5},
        {'params': ftNet.features.denseblock4.parameters(),  'lr': 3e-5},
        {'params': ftNet.features.norm5.parameters(),        'lr': 3e-5},
        {'params': ftNet.classifier.parameters(),            'lr': 1e-4},
    ], weight_decay=1e-2)
    warmup_s2 = lr_scheduler.LinearLR(optimizer_ft_s2, start_factor=0.1, end_factor=1.0, total_iters=3)
    cosine_s2 = lr_scheduler.CosineAnnealingLR(optimizer_ft_s2, T_max=12)
    scheduler_ft_s2 = lr_scheduler.SequentialLR(optimizer_ft_s2,
                        schedulers=[warmup_s2, cosine_s2], milestones=[3])
else:
    print(f'Stage 1 TTA AUC {auc_ft_s1_tta:.4f} < 0.752. Skipping Stage 2.')
    optimizer_ft_s2 = scheduler_ft_s2 = None

Stage 1 TTA AUC 0.7731 >= 0.752. Proceeding to Stage 2.
Stage 2 trainable params: 5,525,249


In [25]:
if optimizer_ft_s2 is not None:
    random.seed(42); npr.seed(42); torch.manual_seed(42); torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
    ftNet_s2 = train_model(ftNet, criterion, optimizer_ft_s2, scheduler_ft_s2,
                           num_epochs=15, patience=5, label_smoothing=0.05)

    auc_ft_s2_tta = eval_val_auc(ftNet_s2, 'DenseNet121 Stage 2', tta=True)

    if auc_ft_s2_tta > auc_ft_s1_tta:
        print(f'Stage 2 better ({auc_ft_s2_tta:.4f} > {auc_ft_s1_tta:.4f}). Using Stage 2.')
        torch.save(ftNet_s2.state_dict(), 'best_densenet121_s2.pth')
        best_ftNet   = ftNet_s2
        auc_ft_final = auc_ft_s2_tta
    else:
        print(f'Stage 2 did NOT improve ({auc_ft_s2_tta:.4f} <= {auc_ft_s1_tta:.4f}). Reverting to Stage 1.')
        ftNet.load_state_dict(torch.load('best_densenet121_s1.pth', map_location=device))
        best_ftNet   = ftNet
        auc_ft_final = auc_ft_s1_tta
else:
    best_ftNet   = ftNet
    auc_ft_final = auc_ft_s1_tta

print(f'Best ftNet val AUC (TTA-4): {auc_ft_final:.4f}')

Epoch 0/14
----------
train Loss: 0.6212  AUC: 0.9689
val Loss: 0.9700  AUC: 0.7521

Epoch 1/14
----------
train Loss: 0.6451  AUC: 0.9584
val Loss: 0.9232  AUC: 0.7567

Epoch 2/14
----------
train Loss: 0.8685  AUC: 0.8132
val Loss: 0.9464  AUC: 0.7624

Epoch 3/14
----------
train Loss: 0.8413  AUC: 0.8342
val Loss: 0.8940  AUC: 0.7688

Epoch 4/14
----------
train Loss: 0.8081  AUC: 0.8508
val Loss: 0.8702  AUC: 0.7761

Epoch 5/14
----------
train Loss: 0.7610  AUC: 0.8758
val Loss: 0.9062  AUC: 0.7618

Epoch 6/14
----------
train Loss: 0.7713  AUC: 0.8677
val Loss: 0.9766  AUC: 0.7549

Epoch 7/14
----------
train Loss: 0.7202  AUC: 0.8911
val Loss: 0.8582  AUC: 0.7765

Epoch 8/14
----------
train Loss: 0.7062  AUC: 0.9002
val Loss: 0.9324  AUC: 0.7704

Epoch 9/14
----------
train Loss: 0.6788  AUC: 0.9093
val Loss: 0.8691  AUC: 0.7707

Epoch 10/14
----------
train Loss: 0.6660  AUC: 0.9172
val Loss: 0.9282  AUC: 0.7623

Epoch 11/14
----------
train Loss: 0.6445  AUC: 0.9249
val Loss:

---
## 13. Generate Test Outputs & Submit

output_custom.csv = w × SE-CustomNetV2 + (1-w) × CustomNetV3 (TTA-4). No DenseNet blend — valid CUSTOM.

output_ft.csv = best DenseNet121 stage (TTA-4).

In [26]:
print('=== Final validation AUC check ===')
print(f'SE-CustomNetV2      (TTA-4): {auc_se:.4f}')
print(f'CustomNetV3         (TTA-4): {auc_v3:.4f}')
print(f'Custom ensemble w={best_w:.1f} (TTA-4): {auc_custom_final:.4f}')
print(f'Best ftNet          (TTA-4): {auc_ft_final:.4f}')

=== Final validation AUC check ===
SE-CustomNetV2      (TTA-4): 0.7716
CustomNetV3         (TTA-4): 0.7656
Custom ensemble w=0.5 (TTA-4): 0.7758
Best ftNet          (TTA-4): 0.8052


In [27]:
test_scores_se = test_model(model_se, tta=True)
test_scores_v3 = test_model(model_v3, tta=True)
outputs_custom = best_w * test_scores_se + (1 - best_w) * test_scores_v3

outputs_ft_submit = test_model(best_ftNet, tta=True)

assert outputs_custom.shape    == (1000, 1)
assert outputs_ft_submit.shape == (1000, 1)
assert np.isfinite(outputs_custom).all()
assert np.isfinite(outputs_ft_submit).all()
print('Checks passed.')
print(f'Custom score range: [{outputs_custom.min():.4f}, {outputs_custom.max():.4f}]')
print(f'FT     score range: [{outputs_ft_submit.min():.4f}, {outputs_ft_submit.max():.4f}]')

Checks passed.
Custom score range: [0.4142, 0.9999]
FT     score range: [0.0289, 0.9981]


In [28]:
with open('output_custom.csv', mode='w', newline='') as f:
    csv.writer(f).writerows(outputs_custom)
with open('output_ft.csv', mode='w', newline='') as f:
    csv.writer(f).writerows(outputs_ft_submit)
print('Written: output_custom.csv  output_ft.csv')

Written: output_custom.csv  output_ft.csv


In [29]:
with ZipFile('./codabench_submission.zip', 'w') as zf:
    zf.write('./output_custom.csv')
    zf.write('./output_ft.csv')
print('Created: codabench_submission.zip')
print(f'\nFinal val AUC summary (eighth_attempt.ipynb):')
print(f'  CUSTOM  SE-CustomNetV2          (TTA-4): {auc_se:.4f}')
print(f'  CUSTOM  CustomNetV3             (TTA-4): {auc_v3:.4f}')
print(f'  CUSTOM  ensemble w={best_w:.1f}         (TTA-4): {auc_custom_final:.4f}')
print(f'  FT      DenseNet121 best stage  (TTA-4): {auc_ft_final:.4f}')

Created: codabench_submission.zip

Final val AUC summary (eighth_attempt.ipynb):
  CUSTOM  SE-CustomNetV2          (TTA-4): 0.7716
  CUSTOM  CustomNetV3             (TTA-4): 0.7656
  CUSTOM  ensemble w=0.5         (TTA-4): 0.7758
  FT      DenseNet121 best stage  (TTA-4): 0.8052
